# CIFAR-10: robustness-focused training, version 3

New experimental recipe targeting the prior reported **93.3% public / 92.9% private** score and **>97.8% clean validation**. These are targets, not achieved results. Full competition training has not been run.

## Changes and research

The prior recipe treated every augmented view independently. This version adds a Jensen-Shannon consistency objective between one clean crop and two independent mixtures of photometric transformations, on every fourth batch. Other batches retain Mixup/CutMix. The last 40 epochs use lighter augmentation and ordinary supervised learning. The 500-epoch schedule and 4,480,474-parameter residual CNN remain. Three views use the SAME model, not a three-model ensemble.

This is an **AugMix-inspired custom recipe**, not the published AugMix algorithm: its operation set and frequency differ. The paper supports testing consistency for robustness, not a claim of improved performance on this competition. Gamma, quantization and fixed-noise validation tests broaden the old brightness/blur-only proxy. All validation transformations use the held-out training split; no hidden test labels or test-based tuning is used.

Research reviewed:
- AugMix (ICLR 2020): https://arxiv.org/abs/1912.02781 and https://github.com/google-research/augmix . Supports the consistency objective and mixtures; no source code or pretrained weights are downloaded by this notebook.
- TrivialAugment (ICCV 2021): https://arxiv.org/abs/2103.10158 . A credible alternative for a later controlled augmentation comparison; not implemented here.
- SAM (ICLR 2021): https://arxiv.org/abs/2010.01412 . A credible alternative optimizer; not stacked into this run because it would confound the robustness comparison and introduce a second optimization change.

No paper establishes a score for this private competition, and their larger-model results do not transfer directly to this parameter budget. More elapsed training time is not evidence of better accuracy.

## Run and reproduce

Import into Kaggle, attach the competition data, select a GPU, and run all cells with Internet off. On NYU HPC set `DATA_ROOT` and `OUT` to your dataset and persistent output directories. Restart the kernel first. The batch now contains up to three times as many images on consistency steps; lower batch_size BEFORE a fresh run if GPU memory is insufficient.

Save the entire output folder, including the notebook, weights, `run.json`, `requirements-runtime.txt`, logs and split. For interruption recovery set `RESUME` to `last.pt`, keeping the original configuration. For prediction only, set `MODE="predict"` and `INFERENCE_CHECKPOINT` to `best_balanced.pt`. Old version-2 checkpoints are intentionally incompatible with this changed recipe.

For a controlled comparison, train a separate run with `consistency_every=0`, retaining the same split, seed and settings; use different output folders. Compare clean and stress holdout metrics before deciding which one-model checkpoint to submit. Never choose a checkpoint by agreement with the old submission CSV. A final Kaggle evaluation is necessary to establish whether the prior score was matched.

## Rules

Competition-only training data, random initialization, residual CNN, no external weights/teachers, no hidden-label reconstruction, one final model below 5M parameters. The prior uploaded prediction CSV is not a training target. CSV ordering follows sample_submission.csv. Sources and complete configuration are retained for the required GitHub submission.

## Verification

Synthetic CPU functionality and reproduction tests are used to check implementation only. They cannot establish CIFAR or Kaggle accuracy. Deterministic operations, saved RNGs, data hashes, environment checks and checkpoint-to-CSV hashes support repeatability in the same environment.


In [ ]:
from pathlib import Path
import os, sys, json, math, random, time, copy, hashlib, platform
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import numpy as np
import pandas as pd
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
import importlib.metadata
import torch
import torch.nn as nn
import torch.nn.functional as F

CFG = dict(seed=0, split_seed=0, holdout=2000, architecture="wrn",
           consistency_every=4, consistency_weight=6.0, epochs=500, batch_size=128, eval_batch_size=256, lr=0.1,
           weight_decay=5e-4, warmup_epochs=10, final_epochs=40,
           label_smoothing=0.05, ema_decay=0.999, eval_every=5,
           deterministic=True, normalization="image", smoke_test=False)
# Set DATA_ROOT if several datasets contain train_labels.csv.
DATA_ROOT = None
OUT = Path("/kaggle/working/reproducible_cifar10") if Path("/kaggle").exists() else Path("./reproducible_cifar10")
# To resume, attach the previous output directory and point here to last.pt.
RESUME = None
# "predict" regenerates predictions from YOUR saved trained checkpoint, without retraining.
MODE = "train"
INFERENCE_CHECKPOINT = None
# Optional planned stop, e.g. 100 epochs this session while retaining the 500-epoch schedule.
RUN_FOR_EPOCHS = None
PIPELINE_VERSION = "cifar10-consistency-3"
STRICT_REPRODUCTION = True
assert MODE in ("train", "predict")
if MODE == "predict":
    if INFERENCE_CHECKPOINT is None: raise ValueError("Set INFERENCE_CHECKPOINT to your own best_balanced.pt")
    inference_state = torch.load(INFERENCE_CHECKPOINT, map_location="cpu", weights_only=False)
    CFG = dict(inference_state["metadata"]["config"])

OUT.mkdir(parents=True, exist_ok=True)

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_all(CFG["seed"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu" and not CFG["smoke_test"] and MODE == "train":
    raise RuntimeError("Enable a GPU for training, or set smoke_test=True for a CPU functionality check.")
torch.backends.cudnn.benchmark = not CFG["deterministic"]
torch.backends.cudnn.deterministic = CFG["deterministic"]
torch.use_deterministic_algorithms(CFG["deterministic"])
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

def environment_signature():
    return dict(python=platform.python_version(),
                packages={name: importlib.metadata.version(name) for name in ("torch","numpy","pandas","Pillow")},
                cuda=torch.version.cuda,cudnn=torch.backends.cudnn.version(),device=DEVICE,
                gpu=torch.cuda.get_device_name(0) if DEVICE=="cuda" else "cpu")

def check_reproduction(saved):
    if STRICT_REPRODUCTION:
        if saved.get("pipeline_version") != PIPELINE_VERSION:
            raise RuntimeError("Checkpoint pipeline version differs. Use its original notebook for exact reproduction.")
        if saved.get("environment") != environment_signature():
            raise RuntimeError("Checkpoint environment differs. Restore the recorded software/GPU environment for exact reproduction, or set STRICT_REPRODUCTION=False to accept an unverified cross-environment run.")
print("Device:", DEVICE, "PyTorch:", torch.__version__)


In [ ]:
if DATA_ROOT is None:
    base = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path("./data")
    hits = sorted(base.rglob("train_labels.csv"))
    if len(hits) != 1: raise RuntimeError(f"Set DATA_ROOT explicitly; found {len(hits)} train_labels.csv files.")
    ROOT = hits[0].parent
else:
    ROOT = Path(DATA_ROOT)
tr = pd.read_csv(ROOT / "train_labels.csv")
te = pd.read_csv(ROOT / "test.csv")
ss = pd.read_csv(ROOT / "sample_submission.csv")
assert {"id", "filename", "label"}.issubset(tr.columns)
assert {"id", "filename"}.issubset(te.columns)
assert list(ss.columns) == ["id", "label"]
assert te.id.is_unique and ss.id.is_unique and tr.id.is_unique
assert set(ss.id) == set(te.id)
classes = sorted(tr.label.unique().tolist())
assert classes == list(range(10)), "Competition labels must be integers 0 through 9"
class_map = {value: i for i, value in enumerate(classes)}
labels = tr.label.map(class_map).to_numpy(dtype=np.int64)

def load_images(folder, names):
    def read(name):
        p = ROOT / folder / str(name)
        if not p.is_file():
            options = [Path(str(p) + ext) for ext in (".png", ".jpg", ".jpeg")]
            p = next((q for q in options if q.is_file()), p)
        with Image.open(p) as im:
            a = np.asarray(im.convert("RGB"), dtype=np.uint8)
        if a.shape != (32,32,3): raise ValueError(f"Unexpected shape {a.shape}: {p}")
        return a
    with ThreadPoolExecutor(max_workers=8) as pool:
        return np.stack(list(pool.map(read, names)))
X = load_images("train_images", tr.filename.tolist())
# Explicitly join by id so sample order can differ from test.csv order.
test_names = te.set_index("id").loc[ss.id, "filename"].tolist()
Xtest = load_images("test_images", test_names)
perm = np.random.RandomState(CFG["split_seed"]).permutation(len(X))
valid_idx, train_idx = perm[:CFG["holdout"]], perm[CFG["holdout"]:]
assert len(set(train_idx) & set(valid_idx)) == 0
assert len(train_idx) + len(valid_idx) == len(X)
if CFG["smoke_test"]:
    train_idx, valid_idx = train_idx[:32], valid_idx[:20]
    CFG.update(epochs=2, warmup_epochs=1, final_epochs=1, batch_size=8, eval_every=1)
np.savez(OUT / "split.npz", train_idx=train_idx, valid_idx=valid_idx)
# Validation split is saved before any model selection.

def to_tensor(a):
    return torch.from_numpy(a.copy()).permute(0,3,1,2).contiguous().to(DEVICE)
Xt, Xv, Xe = to_tensor(X[train_idx]), to_tensor(X[valid_idx]), to_tensor(Xtest)
Yt = torch.as_tensor(labels[train_idx], device=DEVICE)
Yv = torch.as_tensor(labels[valid_idx], device=DEVICE)
train_float = X[train_idx].astype(np.float32) / 255
mean = torch.tensor(train_float.mean((0,1,2)), device=DEVICE).view(1,3,1,1)
std = torch.tensor(train_float.std((0,1,2)), device=DEVICE).view(1,3,1,1)
del train_float
fingerprints = {name: hashlib.sha256((ROOT/name).read_bytes()).hexdigest()
                for name in ("train_labels.csv", "test.csv", "sample_submission.csv")}
# Hash image bytes as well: filenames alone cannot identify a dataset.
fingerprints["train_pixels"] = hashlib.sha256(X.tobytes()).hexdigest()
fingerprints["test_pixels"] = hashlib.sha256(Xtest.tobytes()).hexdigest()
print("Train / validation / test:", len(Xt), len(Xv), len(Xe))
print("Validation class counts:", np.bincount(labels[valid_idx], minlength=10))
del X, Xtest


In [ ]:
class Block(nn.Module):
    def __init__(s, i, o, st):
        super().__init__()
        s.c1 = nn.Conv2d(i, o, 3, st, 1, bias=False); s.b1 = nn.BatchNorm2d(o)
        s.c2 = nn.Conv2d(o, o, 3, 1, 1, bias=False); s.b2 = nn.BatchNorm2d(o)
        s.sk = None if (st == 1 and i == o) else nn.Sequential(nn.Conv2d(i, o, 1, st, bias=False), nn.BatchNorm2d(o))
    def forward(s, x):
        y = F.relu(s.b1(s.c1(x)), inplace=True); y = s.b2(s.c2(y))
        return F.relu(y + (x if s.sk is None else s.sk(x)), inplace=True)
class Net(nn.Module):
    def __init__(s, w=(64,128,256), nb=(3,3,3), nc=10, drop=0.):
        super().__init__()
        s.stem = nn.Sequential(nn.Conv2d(3, w[0], 3, 1, 1, bias=False), nn.BatchNorm2d(w[0]), nn.ReLU(inplace=True))
        L, i = [], w[0]
        for k, (c, n) in enumerate(zip(w, nb)):
            for b in range(n): L.append(Block(i, c, 2 if (b == 0 and k > 0) else 1)); i = c
        s.layers = nn.Sequential(*L); s.drop = nn.Dropout(drop); s.fc = nn.Linear(i, nc)
    def forward(s, x): return s.fc(s.drop(F.adaptive_avg_pool2d(s.layers(s.stem(x)), 1).flatten(1)))

class WideResidualBlock(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(cin)
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.projection = None if cin == cout and stride == 1 else nn.Conv2d(cin, cout, 1, stride, bias=False)

    def forward(self, x):
        h = F.relu(self.bn1(x))
        shortcut = x if self.projection is None else self.projection(h)
        h = self.conv1(h)
        h = self.conv2(F.dropout(F.relu(self.bn2(h)), p=0.1, training=self.training))
        return h + shortcut

class WideResNet28(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Conv2d(3, 16, 3, 1, 1, bias=False)
        layers = []
        cin = 16
        for stage, cout in enumerate((56, 112, 224)):
            for block in range(4):
                layers.append(WideResidualBlock(cin, cout, 2 if stage > 0 and block == 0 else 1))
                cin = cout
        self.blocks = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(cin)
        self.fc = nn.Linear(cin, 10)
        for layer in self.modules():
            if isinstance(layer, nn.Conv2d):
                nn.init.kaiming_normal_(layer.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(layer, nn.BatchNorm2d):
                nn.init.ones_(layer.weight)
                nn.init.zeros_(layer.bias)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x):
        x = self.blocks(self.stem(x))
        x = F.adaptive_avg_pool2d(F.relu(self.bn(x)), 1).flatten(1)
        return self.fc(x)


model = (WideResNet28() if CFG["architecture"] == "wrn" else Net(drop=0.1)).to(DEVICE)
model = model.to(memory_format=torch.channels_last)
nparam = sum(p.numel() for p in model.parameters())
assert nparam <= 5_000_000, nparam
ema = copy.deepcopy(model).eval().requires_grad_(False)
print("Architecture:", CFG["architecture"], "Parameters:", nparam)


In [ ]:
def crop(x, p=4):
    N, C, H, W = x.shape; xp = F.pad(x, (p,p,p,p), mode='reflect')
    i = torch.randint(0, 2*p+1, (N,), device=x.device); j = torch.randint(0, 2*p+1, (N,), device=x.device); ar = torch.arange(H, device=x.device)
    r = (i[:,None] + ar)[:,None,:,None]; c = (j[:,None] + ar)[:,None,None,:]
    return xp[torch.arange(N, device=x.device)[:,None,None,None], torch.arange(C, device=x.device)[None,:,None,None], r, c]

def norm(x):
    if CFG["normalization"] == "image":
        # Scalar per image: preserve relative channel colours.
        x = (x - x.mean((1,2,3), keepdim=True)) / x.std((1,2,3), keepdim=True).clamp_min(1/255)
    elif CFG["normalization"] == "dataset":
        x = (x-mean)/std.clamp_min(1/255)
    else: raise ValueError("Unknown normalization")
    return x.contiguous(memory_format=torch.channels_last)

def uniform(x, lo, hi):
    return torch.empty(len(x),1,1,1,device=x.device).uniform_(lo,hi)

def photometric(x, strength=1.0):
    gate = (uniform(x,0,1) < 0.8).float() * strength
    x = x * (1 + gate * uniform(x,-0.25,0.25))
    mu = x.mean((1,2,3),keepdim=True)
    x = (x-mu)*(1+gate*uniform(x,-0.35,0.35))+mu
    gray = x.mean(1,keepdim=True)
    x = gray+(x-gray)*(1+gate*uniform(x,-0.35,0.35))
    blur = F.avg_pool2d(F.pad(x,(1,1,1,1),mode="reflect"),3,1)
    b = (uniform(x,0,1)<0.10).float()*strength*0.5
    x = (1-b)*x+b*blur
    x = x + (uniform(x,0,1)<0.15)*torch.randn_like(x)*0.025*strength
    return x.clamp(0,1)

def spatial(x):
    x=crop(x.float()/255)
    return torch.where(uniform(x,0,1)<0.5,x.flip(3),x)

def mixed_view(x):
    # Custom photometric mixture inspired by AugMix; not its published operation set.
    weights=torch.distributions.Dirichlet(torch.ones(3,device=x.device)).sample((len(x),))
    result=torch.zeros_like(x)
    for j in range(3):
        z=photometric(x)
        gamma=uniform(x,0.7,1.4)
        z=z.clamp_min(1e-6).pow(gamma)
        result+=weights[:,j,None,None,None]*z
    amount=uniform(x,0,1)
    return norm((1-amount)*x+amount*result)

def augment(x,final=False):
    return norm(photometric(spatial(x),0.35 if final else 1.0))

def consistency_loss(logits):
    # Jensen-Shannon divergence: KL(p_view || mean_probability).
    ps=[z.float().softmax(1) for z in logits]
    mean_p=torch.stack(ps).mean(0).clamp_min(1e-7)
    return sum((p*(p.clamp_min(1e-7).log()-mean_p.log())).sum(1).mean() for p in ps)/len(ps)

def mix_batch(x,y,final):
    if final or random.random() < 0.2: return x,y,y,1.0
    idx = torch.randperm(len(x),device=x.device)
    if random.random() < 0.65:
        lam = float(np.random.beta(1,1)); h,w=x.shape[-2:]
        size=math.sqrt(1-lam); rh,rw=int(h*size),int(w*size)
        cy,cx=np.random.randint(h),np.random.randint(w)
        y1,y2=max(cy-rh//2,0),min(cy+rh//2,h)
        x1,x2=max(cx-rw//2,0),min(cx+rw//2,w)
        x=x.clone(); x[:,:,y1:y2,x1:x2]=x[idx,:,y1:y2,x1:x2]
        lam=1-(y2-y1)*(x2-x1)/(h*w)
    else:
        lam=float(np.random.beta(0.2,0.2)); x=lam*x+(1-lam)*x[idx]
    return x,y,y[idx],lam

# Deterministic synthetic stress tests. These do not diagnose the true test shift.
def shift(x, kind):
    if kind == "clean": return x
    if kind == "bright": return ((x-0.5)*1.15+0.54).clamp(0,1)
    if kind == "dark": return ((x-0.5)*0.85+0.46).clamp(0,1)
    if kind == "blur":
        return 0.5*x+0.5*F.avg_pool2d(F.pad(x,(1,1,1,1),mode="reflect"),3,1)
    if kind == "gamma": return x.clamp_min(1e-6).pow(0.7)
    if kind == "quantize": return (x*31).round()/31
    if kind == "noise":
        # Local generator avoids consuming the training RNG or changing by batch position.
        g=torch.Generator(device=x.device).manual_seed(731)
        return (x+torch.randn(x.shape,device=x.device,generator=g)*0.035).clamp(0,1)
    raise ValueError(kind)

@torch.inference_mode()
def predict(net, data, kind="clean"):
    net.eval(); probs=[]
    for start in range(0,len(data),CFG["eval_batch_size"]):
        x=norm(shift(data[start:start+CFG["eval_batch_size"]].float()/255,kind))
        with torch.autocast(device_type=DEVICE,dtype=torch.float16,enabled=DEVICE=="cuda"):
            p=(net(x).float().softmax(1)+net(x.flip(3)).float().softmax(1))/2
        probs.append(p.cpu())
    return torch.cat(probs)

def evaluate(net):
    metrics={}
    for kind in ("clean","bright","dark","blur","gamma","quantize","noise"):
        p=predict(net,Xv,kind)
        metrics[kind]=(p.argmax(1)==Yv.cpu()).float().mean().item()
    metrics["balanced"] = 0.5*metrics["clean"]+0.5*np.mean([metrics[k] for k in ("bright","dark","blur","gamma","quantize","noise")])
    return metrics


In [ ]:
decay, no_decay = [], []
for p in model.parameters(): (no_decay if p.ndim<=1 else decay).append(p)
optimizer = torch.optim.SGD([dict(params=decay,weight_decay=CFG["weight_decay"]),
                             dict(params=no_decay,weight_decay=0)],
                            lr=CFG["lr"],momentum=0.9,nesterov=True)
scaler = torch.amp.GradScaler("cuda",enabled=DEVICE=="cuda")
criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])
steps_per_epoch = math.ceil(len(Xt)/CFG["batch_size"])

def learning_rate(epoch, batch):
    t=epoch+batch/steps_per_epoch
    if t<CFG["warmup_epochs"]:
        return CFG["lr"]*(0.05+0.95*t/CFG["warmup_epochs"])
    progress=(t-CFG["warmup_epochs"])/max(1,CFG["epochs"]-CFG["warmup_epochs"])
    return 1e-5+0.5*(CFG["lr"]-1e-5)*(1+math.cos(math.pi*min(progress,1)))

@torch.no_grad()
def update_ema(step):
    # A gradual start avoids the early EMA accuracy collapse in the old notebook.
    d=min(CFG["ema_decay"],(1+step)/(10+step))
    for q,p in zip(ema.parameters(),model.parameters()): q.lerp_(p,1-d)
    # Copy BatchNorm statistics rather than averaging already averaged statistics.
    for q,p in zip(ema.buffers(),model.buffers()): q.copy_(p)

def atomic_save(obj,path):
    temp=path.with_suffix(".tmp"); torch.save(obj,temp); os.replace(temp,path)

def rng_state():
    return dict(python=random.getstate(),numpy=np.random.get_state(),torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if DEVICE=="cuda" else [])

def restore_rng(state):
    random.setstate(state["python"]); np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"].cpu())
    if DEVICE=="cuda": torch.cuda.set_rng_state_all([s.cpu() for s in state["cuda"]])

metadata=dict(config=CFG,parameters=nparam,classes=classes,
              pipeline_version=PIPELINE_VERSION,environment=environment_signature(),
              normalization_mean=mean.flatten().tolist(),normalization_std=std.flatten().tolist(),
              fingerprints=fingerprints,python=platform.python_version(),torch=torch.__version__,
              cuda=torch.version.cuda,cudnn=torch.backends.cudnn.version(),
              gpu=torch.cuda.get_device_name(0) if DEVICE=="cuda" else "cpu",
              note="New unverified recipe. No claim of reproducing the lost final model.")
(OUT/("run.json" if MODE=="train" else "inference_run.json")).write_text(json.dumps(metadata,indent=2))
env = environment_signature()
(OUT/"requirements-runtime.txt").write_text("\n".join(f"{name}=={version}" for name,version in env["packages"].items())+"\n")
if MODE=="predict":
    check_reproduction(inference_state["metadata"])
    assert inference_state["metadata"]["fingerprints"]==fingerprints, "Dataset differs from trained checkpoint."
    assert inference_state["metadata"]["parameters"]==nparam
start_epoch,global_step=0,0
best=dict(clean=-1.0,balanced=-1.0); history=[]
if RESUME and MODE=="train":
    # Load only your own trusted checkpoint files.
    ck=torch.load(RESUME,map_location=DEVICE,weights_only=False)
    check_reproduction(ck["metadata"])
    assert ck["metadata"]["config"]==CFG, "Resume config differs; use the original full schedule."
    assert ck["metadata"]["fingerprints"]==fingerprints, "Dataset changed."
    assert np.array_equal(ck["train_idx"],train_idx) and np.array_equal(ck["valid_idx"],valid_idx)
    model.load_state_dict(ck["model"]); ema.load_state_dict(ck["ema"])
    optimizer.load_state_dict(ck["optimizer"]); scaler.load_state_dict(ck["scaler"])
    start_epoch=ck["epoch"]+1; global_step=ck["step"]; best=ck["best"]; history=ck["history"]
    # Keep the historical winners too when resuming into a new working directory.
    import shutil
    for name in ("best_clean.pt","best_balanced.pt"):
        origin=Path(RESUME).parent/name
        if origin.resolve()!=(OUT/name).resolve():
            if not origin.exists(): raise FileNotFoundError(f"Resume needs {origin}")
            shutil.copy2(origin,OUT/name)
    restore_rng(ck["rng"])


In [ ]:
if MODE=="train":
    started=time.time()
    stop_epoch = min(CFG["epochs"], start_epoch + RUN_FOR_EPOCHS) if RUN_FOR_EPOCHS else CFG["epochs"]
    for epoch in range(start_epoch,stop_epoch):
        epoch_started=time.time()
        model.train(); order=torch.randperm(len(Xt),device=DEVICE)
        total_loss=0.; count=0
        final=epoch>=CFG["epochs"]-CFG["final_epochs"]
        for b,start in enumerate(range(0,len(Xt),CFG["batch_size"])):
            ids=order[start:start+CFG["batch_size"]]
            y=Yt[ids]
            use_consistency=(not final and CFG["consistency_every"]>0 and b%CFG["consistency_every"]==0)
            if use_consistency:
                base=spatial(Xt[ids])
                views=(norm(base),mixed_view(base),mixed_view(base))
            else:
                x=augment(Xt[ids],final)
                x,ya,yb,lam=mix_batch(x,y,final)
            lr=learning_rate(epoch,b)
            for group in optimizer.param_groups: group["lr"]=lr
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE,dtype=torch.float16,enabled=DEVICE=="cuda"):
                if use_consistency:
                    # A shared forward pass updates BatchNorm once for all three views.
                    logits=model(torch.cat(views)).chunk(3)
                    loss=criterion(logits[0],y)+CFG["consistency_weight"]*consistency_loss(logits)
                else:
                    logits=model(x)
                    loss=lam*criterion(logits,ya)+(1-lam)*criterion(logits,yb)
            if not torch.isfinite(loss): raise FloatingPointError(f"Non-finite loss at epoch {epoch+1}")
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(),5.)
            old_scale=scaler.get_scale()
            scaler.step(optimizer); scaler.update()
            if scaler.get_scale()>=old_scale:
                global_step+=1; update_ema(global_step)
            total_loss+=loss.item()*len(ids); count+=len(ids)
        row=dict(epoch=epoch+1,loss=total_loss/count,lr=lr,seconds=time.time()-started)
        if (epoch+1)%CFG["eval_every"]==0 or epoch==0 or epoch==CFG["epochs"]-1:
            for source,net in (("raw",model),("ema",ema)):
                metrics=evaluate(net)
                row.update({f"{source}_{k}":float(v) for k,v in metrics.items()})
                for selection in ("clean","balanced"):
                    score=float(metrics[selection])
                    if score>best[selection]:
                        best[selection]=score
                        atomic_save(dict(model=net.state_dict(),epoch=epoch+1,source=source,
                                         validation=metrics,metadata=metadata),OUT/f"best_{selection}.pt")
        row["epoch_seconds"]=time.time()-epoch_started
        history.append(row)
        pd.DataFrame(history).to_csv(OUT/"history.csv",index=False)
        atomic_save(dict(model=model.state_dict(),ema=ema.state_dict(),optimizer=optimizer.state_dict(),
                         scaler=scaler.state_dict(),epoch=epoch,step=global_step,best=best,history=history,
                         metadata=metadata,train_idx=train_idx,valid_idx=valid_idx,rng=rng_state()),OUT/"last.pt")
        print(json.dumps(row),flush=True)
    if stop_epoch<CFG["epochs"]:
        print("Planned pause. Save the WHOLE output folder, set RESUME to last.pt, keep epochs unchanged.")
    else:
        print("Training complete. Best clean / balanced selection scores:",best)


In [ ]:
results={}
selections=("clean","balanced") if MODE=="train" else ("checkpoint",)
for selection in selections:
    ck=(torch.load(OUT/f"best_{selection}.pt",map_location=DEVICE,weights_only=False)
        if MODE=="train" else inference_state)
    model.load_state_dict(ck["model"])
    vp=predict(model,Xv); tp=predict(model,Xe)
    np.save(OUT/f"validation_probs_{selection}.npy",vp.numpy())
    np.save(OUT/f"test_probs_{selection}.npy",tp.numpy())
    audit=tr.iloc[valid_idx][["id","filename","label"]].copy()
    audit["prediction"]=[classes[i] for i in vp.argmax(1).tolist()]
    audit.to_csv(OUT/f"validation_predictions_{selection}.csv",index=False)
    sub=ss.copy(); sub["label"]=[classes[i] for i in tp.argmax(1).tolist()]
    assert sub.id.equals(ss.id) and len(sub)==len(te) and not sub.isna().any().any()
    assert set(sub.label).issubset(set(classes))
    sub.to_csv(OUT/f"submission_{selection}.csv",index=False)
    measured=(vp.argmax(1)==Yv.cpu()).float().mean().item()
    results[selection]=dict(epoch=ck["epoch"],source=ck["source"],
                            clean_validation=measured,stress_validation=ck["validation"],
                            target_exceeded=measured>0.978,public_accuracy=None,private_accuracy=None)
# One checkpoint and its flip view; no multi-model ensemble.
import shutil
shutil.copy2(OUT/("submission_balanced.csv" if MODE=="train" else "submission_checkpoint.csv"),OUT/"submission.csv")
if Path("/kaggle/working").exists():
    shutil.copy2(OUT/"submission.csv","/kaggle/working/submission.csv")
(OUT/"results.json").write_text(json.dumps(results,indent=2))
print(json.dumps(results,indent=2))
prediction_file=OUT/"submission.csv"
(OUT/"submission.sha256").write_text(hashlib.sha256(prediction_file.read_bytes()).hexdigest()+"  submission.csv\n")
print("CSV ready:",prediction_file)
print("SHA256:",hashlib.sha256(prediction_file.read_bytes()).hexdigest())
print("Public/private scores remain unknown until Kaggle evaluation.")
if CFG["smoke_test"]: print("SMOKE TEST ONLY: these weights/CSVs are not trained competition results.")


## Using the outputs

- `submission.csv`: balanced-checkpoint predictions in the exact sample-submission order. Upload this file manually to Kaggle.
- `submission_clean.csv`: alternative selected solely by clean validation; do not assume it is better on shifted images. Choose your selection policy before looking at leaderboard feedback.
- `best_clean.pt` / `best_balanced.pt`: standalone weights with validation metrics and configuration. Each file contains one model under 5M parameters.
- `last.pt`: resume state, including optimizer, scaler, RNGs and split. To resume after a time limit, attach the whole previous output folder, set `RESUME`, and retain the original epoch total. Changing from 500 to 800 midway changes the schedule and is a new experiment.
- `run.json`, `history.csv`, `split.npz`, `results.json`, validation predictions and probabilities: audit trail. Archive all of these with the CSV and notebook.

Set `smoke_test=True` to check loading/training/export on a small subset first; switch it back and restart the kernel for the full run. Batch size 128 and inference batch size 256 are conservative defaults. Lower them if needed before the full run; that changes the experiment. Kaggle session limits may require resuming. Merely increasing epochs does not establish the requested accuracy.

The brightness/contrast explanation in the report is a hypothesis. Average pixel statistics do not establish the cause of the leaderboard gap. This notebook uses fixed, training-defined stress tests; it does not access test labels, retrieve external CIFAR images, use pretrained weights, or tune augmentation to the test set.

Architecture: https://arxiv.org/abs/1605.07146 and https://arxiv.org/abs/1512.03385
CutMix: https://arxiv.org/abs/1905.04899 ; Mixup: https://arxiv.org/abs/1710.09412
PyTorch AMP: https://docs.pytorch.org/docs/stable/amp.html
Reproducibility: https://docs.pytorch.org/docs/stable/notes/randomness.html
Pinned environment versions are recorded by the run; identical seeds do not guarantee identical results across GPU types or PyTorch versions. No full accuracy benchmark has been run for this newly constructed recipe.
